# SysML v2 -> ArangoDB -> natural-language answers

```bash
docker run -d --name christian-webb-drone-arango -p 8529:8529 \
  -e ARANGO_ROOT_PASSWORD=testpass arangodb:3.12.9.4 \
  arangod --experimental-vector-index=true
python build.py
```

In [ ]:
import json, os, sys
from pathlib import Path

from sysml import config, nl
from sysml.pipeline import enrich, project

db = config.db()
print(f"{config.ARANGO_URL}  db={config.DB_NAME}\n")
for name in config.ALL_COLLECTIONS:
    coll = db.collection(name)
    kind = "edge" if coll.properties()["edge"] else "document"
    print(f"  {coll.count():>6}  {name:<20} {kind}")

http://localhost:8529  db=dronegraph

      30  sysml_Documents      document
     200  sysml_Chunks         document
    2359  sysml_Entities       document
      44  sysml_Communities    document
    9764  sysml_Relations      edge


## 1. Why I Built a Parser

`graphrag_importer`'s normal job is to read prose and ask an LLM to infer entities
and relations from it. Point it at a SysML file and it will do exactly that and reguess
from the text.

The whole point of SysML is that constraints and relations are already hard defined in a
deterministic way, so that seems to defeat the purpose. By making deterministic parse logic,
we can achieve:

- no hallucinated edges, because no edge is inferred
- every element keeps its `sourceFile:sourceLine`, so any answer is quotable to the line

In [2]:
model = json.loads(config.MODEL_JSON.read_text(encoding="utf-8"))
authored = model["authored_relation_counts"]
distinct = {}
for r in model["relations"]:
    distinct[r["type"]] = distinct.get(r["type"], 0) + 1

print(f"{len(model['files'])} .sysml files -> {len(model['elements'])} elements, "
      f"{len(model['relations'])} relations, {len(model['unresolved'])} unresolved refs\n")
print(f"  {'relation':<16}{'authored':>10}{'distinct':>10}")
for kind, n in sorted(authored.items(), key=lambda kv: -kv[1]):
    print(f"  {kind:<16}{n:>10}{distinct.get(kind, 0):>10}")

print("\ncross-check -- count the statements in the sources directly:")
import re
sources = [p.read_text(encoding="utf-8") for p in sorted(config.MODELS.rglob("*.sysml"))]
for pattern, label in [(r"^\s*satisfy\s", "satisfies"), (r"#refinement\s+dependency", "refines"),
                       (r"^\s*subject\s", "subject"), (r"^\s*perform\s|^\s*do\s+action\s", "performs"),
                       (r"^\s*variant\s", "variantOf")]:
    counted = sum(len(re.findall(pattern, text, re.M)) for text in sources)
    verdict = "match" if counted == authored.get(label, 0) else "differs"
    print(f"  {label:<12} in source {counted:>5}   parser authored {authored.get(label, 0):>5}   {verdict}")

30 .sysml files -> 2359 elements, 5300 relations, 0 unresolved refs

  relation          authored  distinct
  owns                  2251      2251
  typedBy                989       989
  specializes            777       777
  refines                369       368
  satisfies              273       263
  redefines              184       184
  imports                167       165
  performs               105       105
  transitionsTo           59        59
  connects                49        46
  valueRef                32        32
  sliceOf                 31        31
  subject                 16        16
  variantOf               10        10
  exhibits                 2         2
  derives                  2         2

cross-check -- count the statements in the sources directly:
  satisfies    in source   273   parser authored   273   match
  refines      in source   369   parser authored   369   match
  subject      in source    16   parser authored    16   match
  performs     in

## 2. Translating into graphrag_importer's schema

| importer slot | filled with | why |
|---|---|---|
| `Document` | one `.sysml` file | the provenance root |
| `Chunk` | a window of source text cut at declaration boundaries | the retrievable text unit; chunks tile each file with no gaps |
| `Entity` | one SysML element, rendered as prose | the embeddable unit |
| `Relation` | every edge | typed, weighted, and embeddable |
| `Community` | a cluster with a generated report | the unit a global question is answered from |

Five slots, five things pulled straight back out of the graph below -- one telling
example of each, with what it is there to show.

In [3]:
E, R, C = config.ENTITIES, config.RELATIONS, config.COMMUNITIES
one = lambda aql, **b: next(db.aql.execute(aql, bind_vars=b or None))

print("=" * 78)
print("1. Document -- the provenance root. No text of its own; it exists so every")
print("   answer can name the file it came from.")
print("=" * 78)
d = one(f"FOR d IN {config.DOCUMENTS} FILTER d.file_name LIKE '%TechnicalComponentsPackage%' RETURN d")
n_chunks = one(f"RETURN LENGTH(FOR c IN {config.CHUNKS} FILTER c.file_name == @f RETURN 1)",
               f=d["file_name"])
print(f"   file_name : {d['file_name']}")
print(f"   file_ids  : {d['file_ids']}")
print(f"   -> {n_chunks} chunks hang off this one document")

print("\n" + "=" * 78)
print("2. Chunk -- the retrievable text unit. Cut at declaration boundaries, never")
print("   mid-declaration, and the chunks tile a file with no gaps and no overlap.")
print("=" * 78)
rows = list(db.aql.execute(
    f"FOR c IN {config.CHUNKS} FILTER c.file_name == @f SORT c.start_line "
    "RETURN {i: c.chunk_order_index, s: c.start_line, e: c.end_line, t: c.tokens}",
    bind_vars={"f": d["file_name"]}))
for r in rows[:5]:
    print(f"   chunk {r['i']:<3} lines {r['s']:>5}-{r['e']:<5} ~{r['t']:>4} tokens")
gaps = [(a["e"], b["s"]) for a, b in zip(rows, rows[1:]) if b["s"] != a["e"] + 1]
print(f"   -> {len(rows)} chunks covering lines {rows[0]['s']}-{rows[-1]['e']}, gaps/overlaps: {len(gaps)}")

print("\n" + "=" * 78)
print("3. Entity -- one SysML element. The `description` is generated prose and is")
print("   what gets embedded, so retrieval matches on meaning rather than on name.")
print("=" * 78)
e = one(f"FOR e IN {E} FILTER e.entity_name == 'CapabilitiesPackage::MultiStagePropulsion' RETURN e")
print(f"   entity_name : {e['entity_name']}")
print(f"   entity_type : {e['entity_type']}   (the real SysML metatype, not a guess)")
print(f"   at          : {e['source_file']}:{e['source_line']}")
print(f"   description : {e['description'][:240]}...")

print("\n" + "=" * 78)
print("4. Relation -- every edge. `type` is the importer's vocabulary, and the")
print("   authored SysML relation rides on `relationship_type`.")
print("=" * 78)
for r in db.aql.execute(f"""FOR r IN {R} FILTER r.relationship_type IN
    ['satisfies', 'refines', 'performs', 'subject', 'valueRef']
    COLLECT rt = r.relationship_type INTO g
    LET s = g[0].r
    RETURN {{rt, type: s.type, text: s.description}}"""):
    print(f"   {r['rt']:<11} type={r['type']:<12} {r['text'][:58]}")

print("\n" + "=" * 78)
print("5. Community -- the unit a whole-model question is answered from. A cluster")
print("   plus one generated report, embedded like any other text.")
print("=" * 78)
c = one(f"FOR c IN {C} FILTER c.level == 0 SORT c.occurrence DESC LIMIT 1 RETURN c")
print(f"   title      : {c['title']}")
print(f"   level      : {c['level']}   members: {c['occurrence']}")
print(f"   report     : {c['report_string'][:240]}...")

1. Document -- the provenance root. No text of its own; it exists so every
   answer can name the file it came from.


   file_name : apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml
   file_ids  : ['sysml:apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml']
   -> 7 chunks hang off this one document

2. Chunk -- the retrievable text unit. Cut at declaration boundaries, never
   mid-declaration, and the chunks tile a file with no gaps and no overlap.
   chunk 0   lines     1-42    ~ 389 tokens
   chunk 1   lines    43-86    ~ 433 tokens
   chunk 2   lines    87-122   ~ 440 tokens
   chunk 3   lines   123-157   ~ 432 tokens
   chunk 4   lines   158-199   ~ 446 tokens
   -> 7 chunks covering lines 1-253, gaps/overlaps: 0

3. Entity -- one SysML element. The `description` is generated prose and is
   what gets embedded, so retrieval matches on meaning rather than on name.


   entity_name : CapabilitiesPackage::MultiStagePropulsion
   entity_type : PartDefinition   (the real SysML metatype, not a guess)
   at          : apollo-11-sysml-v2/Purpose/CapabilitiesPackage.sysml:29
   description : MultiStagePropulsion is a PartDefinition in the apollo-11 model (Purpose layer), declared at apollo-11-sysml-v2/Purpose/CapabilitiesPackage.sysml:29. The ability to efficiently shed mass during ascent through the sequential operation of mul...

4. Relation -- every edge. `type` is the importer's vocabulary, and the
   authored SysML relation rides on `relationship_type`.


   performs    type=RELATED_TO   launchControlCenter performs performCrewIngress
   refines     type=RELATED_TO   PerformPropellantLoading refines LoadConsumablesAndPropell
   satisfies   type=RELATED_TO   load satisfies flr-R001
   subject     type=RELATED_TO   SystemPowerAnalysis has as its subject System
   valueRef    type=RELATED_TO   item1 takes as its value high

5. Community -- the unit a whole-model question is answered from. A cluster
   plus one generated report, embedded like any other text.
   title      : Achieving Lunar Orbit Functions
   level      : 0   members: 111
   report     : Achieving Lunar Orbit Functions

This cluster focuses on the functions necessary to achieve and maintain lunar orbit during the Apollo 11 mission. It includes actions related to propulsive maneuvers, telemetry analysis, and various operatio...


`type` on an edge is a closed five-value vocabulary and exists for validation.
`relationship_type` is the importer's own field for a typed relation, `import_relationships`
writes `data.get("relationship_type", "UNKNOWN")` on every `RELATED_TO` edge.

So a consumer that reads only `type` sees 5,300 identical `RELATED_TO` edges.
One that reads `relationship_type` sees sixteen distinct engineering relations.
That is a limit of the graphrag_importer schema.

In [4]:
print("edge `type` -- the importer's closed vocabulary:")
for r in db.aql.execute(f"FOR r IN {config.RELATIONS} COLLECT t = r.type WITH COUNT INTO n "
                        "SORT n DESC RETURN {t, n}"):
    print(f"  {r['n']:>6}  {r['t']}")

print("\n`relationship_type` on RELATED_TO -- the authored SysML relation:")
for r in db.aql.execute(f"FOR r IN {config.RELATIONS} FILTER r.type == 'RELATED_TO' "
                        "COLLECT t = r.relationship_type WITH COUNT INTO n SORT n DESC RETURN {t, n}"):
    print(f"  {r['n']:>6}  {r['t']}")

edge `type` -- the importer's closed vocabulary:
    5300  RELATED_TO
    2284  MENTIONED_IN
    1944  IN_COMMUNITY
     200  PART_OF
      36  HAS_PARENT

`relationship_type` on RELATED_TO -- the authored SysML relation:


    2251  owns
     989  typedBy
     777  specializes
     368  refines
     263  satisfies
     184  redefines
     165  imports
     105  performs
      59  transitionsTo
      46  connects
      32  valueRef
      31  sliceOf
      16  subject
      10  variantOf
       2  derives
       2  exhibits


### Is the graph well-formed?

Counting edges says the parser produced *something*. These say it produced the right
thing, and each one caught a real bug while this was being built.

**Structural.** No self-loop, no dangling endpoint, no isolated element, no element
with two owners.

**Namespaces.** The three models are independent, so an edge between them is always
a resolution accident.

**Endpoint types.** Every relation should connect the kinds of thing it is supposed
to connect: components satisfy requirements, components perform actions, snapshots
slice parts. 

In [5]:
q = lambda s: list(db.aql.execute(s))
E, R = config.ENTITIES, config.RELATIONS
print("structural invariants (all should be 0):")
for label, aql in [
    ("self-loops", f"RETURN LENGTH(FOR r IN {R} FILTER r._from == r._to RETURN 1)"),
    ("dangling endpoints", f"RETURN LENGTH(FOR r IN {R} FILTER DOCUMENT(r._from) == null "
                           "OR DOCUMENT(r._to) == null RETURN 1)"),
    ("isolated entities", f"RETURN LENGTH(FOR e IN {E} FILTER LENGTH("
                          f"FOR x IN 1..1 ANY e {R} LIMIT 1 RETURN 1) == 0 RETURN 1)"),
    ("elements with 2+ owners", f"RETURN LENGTH(FOR e IN {E} LET p = LENGTH("
                                f"FOR x, ed IN 1..1 INBOUND e {R} "
                                "FILTER ed.relationship_type == 'owns' RETURN 1) FILTER p > 1 RETURN 1)"),
    ("duplicate entity_names", f"RETURN LENGTH(FOR e IN {E} COLLECT n = e.entity_name "
                               "WITH COUNT INTO c FILTER c > 1 RETURN 1)"),
    ("edges across the 3 models", f"RETURN LENGTH(FOR r IN {R} FILTER r.type == 'RELATED_TO' "
                                  "LET a = DOCUMENT(r._from), b = DOCUMENT(r._to) "
                                  "FILTER a.model != b.model AND a.is_library != true "
                                  "AND b.is_library != true RETURN 1)"),
    ("unresolved references", f"RETURN LENGTH(FOR e IN {E} FILTER e.entity_type == 'Unresolved' RETURN 1)"),
]:
    print(f"  {label:<28} {q(aql)[0]}")

print("\nwhat each relation connects (top pattern, and it should read as a sentence):")
for rel in ("satisfies", "refines", "performs", "subject", "valueRef", "variantOf",
            "transitionsTo", "sliceOf", "connects", "derives"):
    rows = q(f"""FOR r IN {R} FILTER r.relationship_type == '{rel}'
        LET a = DOCUMENT(r._from), b = DOCUMENT(r._to)
        COLLECT s = a.entity_type, d = b.entity_type WITH COUNT INTO n
        SORT n DESC LIMIT 1 RETURN {{s, d, n}}""")
    if rows:
        r0 = rows[0]
        print(f"  {rel:<14} {r0['s']} --{rel}--> {r0['d']}   ({r0['n']})")

structural invariants (all should be 0):
  self-loops                   0


  dangling endpoints           0
  isolated entities            0
  elements with 2+ owners      0
  duplicate entity_names       0


  edges across the 3 models    0
  unresolved references        0

what each relation connects (top pattern, and it should read as a sentence):
  satisfies      ActionUsage --satisfies--> RequirementUsage   (155)
  refines        RequirementDefinition --refines--> RequirementDefinition   (214)


  performs       PartDefinition --performs--> ActionUsage   (53)
  subject        AnalysisUsage --subject--> PartDefinition   (5)
  valueRef       ItemUsage --valueRef--> EnumerationUsage   (28)
  variantOf      PartUsage --variantOf--> PartDefinition   (6)
  transitionsTo  ActionUsage --transitionsTo--> ActionUsage   (29)


  sliceOf        SnapshotUsage --sliceOf--> PartUsage   (31)
  connects       PartUsage --connects--> RequirementUsage   (33)
  derives        RequirementUsage --derives--> RequirementUsage   (2)


## 3. What an entity looks like

An `Entity` has to be embeddable, so its `description` is what retrieval actually
matches against.

Since SysML gives us so much information about an entity, we can generate a description
deterministically based on:

what its documentation says (it basically gives us a description already), what it is, where it is,
what its attributes are worth, and what it is attached to.

I think this is simpler than having an LLM write the description, since the LLM would write
basically the same thing, but take more time and introduce potential hallucinations.

Open question: is there something being lost by not having an LLM make the description?

In [6]:
qn = "TechnicalComponentsPackage::S-IC"
e = db.collection(config.ENTITIES).get(project.key_of(qn))
print(f"entity_name : {e['entity_name']}")
print(f"entity_type : {e['entity_type']}")
print(f"at          : {e['source_file']}:{e['source_line']}")
print(f"attributes  : {json.dumps(e['attributes'])}")
print(f"\ndescription (this is the embedded text):\n{e['description']}")

chunk = next(db.aql.execute(
    f"FOR ent IN {config.ENTITIES} FILTER ent.entity_name == @q "
    f"FOR c, edge IN 1..1 OUTBOUND ent {config.RELATIONS} "
    "FILTER edge.type == 'MENTIONED_IN' LIMIT 1 "
    "RETURN {at: CONCAT(c.file_name, ':', c.start_line, '-', c.end_line), content: c.content}",
    bind_vars={"q": qn}))
print(f"\nthe chunk it is declared in -- {chunk['at']}:")
print("\n".join(chunk["content"].splitlines()[:14]))

entity_name : TechnicalComponentsPackage::S-IC
entity_type : PartDefinition
at          : apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml:217
attributes  : {"propellantMass": {"value": 2077000, "unit": "kg", "raw": "2077000 [kg]"}, "dryMass": {"value": 137000, "unit": "kg", "raw": "137000 [kg]"}}

description (this is the embedded text):
S-IC is a PartDefinition in the apollo-11 model (Technical layer), declared at apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml:217. The first stage of the Saturn V, providing the initial ~7.7 million pounds of thrust for liftoff. Attributes: propellantMass = 2077000 kg; dryMass = 137000 kg. It specializes RocketStage. It contains propellantMass, dryMass, upperStagePort, engine1, engine2, engine3, engine4, engine5, engines.

the chunk it is declared in -- apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml:200-242:

	part def 'S-II' :> RocketStage {
		doc /* The second stage of the Saturn V, providing massive thrust

## 4. SysML gives us descriptions of relations

SysML offers us very useful descriptions for each relation, so we can easily query based on the relation between two entities.

In [8]:
for relation in ("satisfies", "refines", "performs"):
    print(f"-- nearest `{relation}` edges to \"guiding the rocket during ascent\"")
    for r in nl.search_relations(db, "guiding the rocket during ascent", relation=relation, k=3):
        print(f"   {r['score']:.3f}  {r['description'][:70]:<70} ({r['at']})")
    print()

-- nearest `satisfies` edges to "guiding the rocket during ascent"


   0.410  lunarModuleAscentStage satisfies clr-R033                              (apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml:157)
   0.403  lunarModuleAscentStage satisfies clr-R115                              (apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml:226)
   0.398  lunarModuleAscentStage satisfies clr-R103                              (apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml:223)

-- nearest `refines` edges to "guiding the rocket during ascent"


   0.517  GuideAtmosphericEntry refines GuideReentryTrajectory                   (apollo-11-sysml-v2/Function/FunctionsPackage.sysml:303)
   0.513  AscentGuidanceRequirement refines TransLunarInjectionAccuracyRequireme (apollo-11-sysml-v2/Requirements/FunctionalRequirementsPackage.sysml:120)
   0.505  LMDSLaunchPlatform refines AscentThrustInitiationRequirement           (apollo-11-sysml-v2/Requirements/TechnicalRequirementsPackage.sysml:311)

-- nearest `performs` edges to "guiding the rocket during ascent"


   0.631  launchVehicle performs guideAscentTrajectory                           (apollo-11-sysml-v2/Execution/Apollo11MissionExecutionPackage.sysml:90)
   0.570  LaunchSystem performs guideAscentTrajectory                            (apollo-11-sysml-v2/Logical/LogicalComponentsPackage.sysml:13)
   0.540  spacecraft performs executeAscentBurn                                  (apollo-11-sysml-v2/Execution/Apollo11MissionExecutionPackage.sysml:366)



## 5. Communities, and what a generated report is good for

The importer runs Leiden because an LLM-extracted graph has no structure to go on.
A SysML model has two, and they say different things:

- **Level 1** is the package structure -- how the engineers themselves grouped the
  model
- **Level 0** is label propagation over the *meaning-bearing* edges only
  (`satisfies`, `refines`, `performs`, `specializes`, `subject`, ...), deliberately
  excluding `owns`. Containment connects the whole model into one blob; the
  traceability edges cut across packages and find the requirement -> capability ->
  function -> component clusters that are actually about something.

Each community gets one LLM-written report from counted facts and member names --
the only text in the whole pipeline an LLM authors. Those reports are embedded, so a
question about the model *as a whole* retrieves a summary instead of trying to
assemble one from two thousand individual elements. Section 8 uses that.

In [9]:
total = db.collection(config.COMMUNITIES).count()
print(f"{total} communities -- the five largest at each level:\n")
for level in (1, 0):
    for c in db.aql.execute(f"FOR c IN {config.COMMUNITIES} FILTER c.level == {level} "
                            "SORT c.occurrence DESC LIMIT 5 "
                            "RETURN {title: c.title, n: c.occurrence}"):
        print(f"  L{level}  {c['n']:>4}  {c['title'][:64]}")
    print()

c = next(db.aql.execute(f"FOR c IN {config.COMMUNITIES} FILTER c.level == 0 "
                        "SORT c.occurrence DESC LIMIT 1 RETURN c"))
print(f"{'=' * 74}\nreport for the largest level-0 cluster ({c['occurrence']} members), first 20 lines"
      f"\n{'=' * 74}")
lines = c["report_string"].splitlines()
print("\n".join(lines[:20]))
print(f"\n... {len(lines) - 20} more lines" if len(lines) > 20 else "")

44 communities -- the five largest at each level:

  L1   249  Apollo 11 Mission Requirements Overview
  L1   230  Apollo-11 Mission Purpose and Stakeholders
  L1   222  Apollo 11 Technical Components and Astronauts
  L1   111  Apollo-11 Functional Operations
  L1    56  Apollo Program Missions Overview

  L0   111  Achieving Lunar Orbit Functions
  L0   104  Apollo 11 Mission Capabilities and Requirements
  L0    71  Apollo 11 Spacecraft Components
  L0    49  Apollo 11 Mission System Overview
  L0    41  American Public Concerns for Apollo 11

report for the largest level-0 cluster (111 members), first 20 lines
Achieving Lunar Orbit Functions

This cluster focuses on the functions necessary to achieve and maintain lunar orbit during the Apollo 11 mission. It includes actions related to propulsive maneuvers, telemetry analysis, and various operational tasks essential for lunar orbit insertion and subsequent mission phases.

- High concentration of ActionDefinitions (110 out of 111 mem

## 6. Asking in English -- AQLizer

The only thing this project gives AQLizer is `aql_examples`, since we need to give it context on how sysml translates to graphrag_importer syntax. For example, `satisfy R by S` is stored as an edge *from* the satisfier.

In [10]:
az = nl.instance()
schema = az.schema
print(f"what AQLizer sees: {len(schema.get('collection_schema', []))} collections sampled, "
      f"{len(schema.get('graph_schema', []))} named graphs")
print(f"domain knowledge given to it: {config.AQL_EXAMPLES.name}, "
      f"{len(az.examples):,} characters, {len(az.examples.splitlines())} lines\n")

az.ask("How many satisfies relations are there in the whole graph, "
       "and how many refines?").show()

LLM provider initialized successfully.


Connecting to ArangoDB at http://localhost:8529 (timeout=300s)


Loaded AQL reference documentation from C:\Users\Christian\Desktop\arango\natural-language-service\txt2aql\prompts\aql_ref.md (104940 chars)


Successfully injected AQL reference into generation prompt.


Successfully injected AQL reference into fix prompt.


what AQLizer sees: 5 collections sampled, 1 named graphs
domain knowledge given to it: aql_examples.md, 11,709 characters, 266 lines



Q  How many satisfies relations are there in the whole graph, and how many refines?

AQL
   WITH sysml_Entities, sysml_Chunks, sysml_Documents, sysml_Communities
   FOR r IN sysml_Relations
     FILTER r.type == "RELATED_TO" AND r.relationship_type IN ["satisfies", "refines"]
     COLLECT relation = r.relationship_type WITH COUNT INTO n
     RETURN {relation, count: n}

rows (2, first 2)
   {"relation": "refines", "count": 368}
   {"relation": "satisfies", "count": 263}

A  In the entire graph, there are 263 "satisfies" relations and 368 "refines" relations.



In [11]:
az.ask("Which requirements in the drone-base model does nothing satisfy?").show()

Loaded AQL reference documentation from C:\Users\Christian\Desktop\arango\natural-language-service\txt2aql\prompts\aql_ref.md (104940 chars)


Successfully injected AQL reference into generation prompt.


Successfully injected AQL reference into fix prompt.


Q  Which requirements in the drone-base model does nothing satisfy?

AQL
   WITH sysml_Documents, sysml_Chunks, sysml_Communities, sysml_Entities
   FOR e IN sysml_Entities
     FILTER e.entity_type IN ["RequirementUsage", "RequirementDefinition"]
     FILTER e.model == "drone-base"
     LET satisfiers = LENGTH(
       FOR r IN sysml_Relations
         FILTER r._to == e._id AND r.relationship_type == "satisfies"
         LIMIT 1 RETURN 1)
     FILTER satisfiers == 0
     RETURN {entity_name: e.entity_name, at: CONCAT(e.source_file, ":", e.source_line)}

rows (3, first 3)
   {"entity_name": "Drone_SystemRequirements::totalMass", "at": "Drone_BaseArchitecture.sysml:30"}
   {"entity_name": "Drone_SystemRequirements::battery", "at": "Drone_BaseArchitecture.sysml:36"}
   {"entity_name": "Drone_SystemRequirements::maxCapacity", "at": "Drone_BaseArchitecture.sysml:39"}

A  The query found that there are three requirements in the drone-base model that currently have nothing to satisfy them. Th

In [12]:
az.ask("What is the total dry mass of the Saturn V, summed from its stages, "
       "and which stage dominates?").show()

Loaded AQL reference documentation from C:\Users\Christian\Desktop\arango\natural-language-service\txt2aql\prompts\aql_ref.md (104940 chars)


Successfully injected AQL reference into generation prompt.


Successfully injected AQL reference into fix prompt.


Q  What is the total dry mass of the Saturn V, summed from its stages, and which stage dominates?

AQL
   WITH sysml_Entities, sysml_Relations
   FOR e IN sysml_Entities
     FILTER e.name == "SaturnV"
     LET parts = (
       FOR child, edge IN 1..6 OUTBOUND e sysml_Relations
         FILTER edge.relationship_type IN ["owns", "typedBy"]
         FILTER child.attributes.dryMass.value != null
         RETURN DISTINCT {name: child.name, mass: child.attributes.dryMass.value,
                          at: CONCAT(child.source_file, ":", child.source_line)})
     LET maxStage = (
       FOR p IN parts
       SORT p.mass DESC
       LIMIT 1
       RETURN p
     )
     RETURN {totalMass: SUM(parts[*].mass), dominantStage: FIRST(maxStage)}

rows (1, first 1)
   {"totalMass": 188650, "dominantStage": {"name": "S-IC", "mass": 137000, "at": "apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml:217"}}

A  The total dry mass of the Saturn V, summed from its stages, is 188,650 kg. The stage

In [13]:
az.ask("Which technical component ends up responsible for GuideAscentTrajectory, "
       "and through which logical component?").show()

Loaded AQL reference documentation from C:\Users\Christian\Desktop\arango\natural-language-service\txt2aql\prompts\aql_ref.md (104940 chars)


Successfully injected AQL reference into generation prompt.


Successfully injected AQL reference into fix prompt.


Q  Which technical component ends up responsible for GuideAscentTrajectory, and through which logical component?

AQL
   WITH sysml_Entities, sysml_Relations
   FOR fn IN sysml_Entities
     FILTER fn.name == "GuideAscentTrajectory"
     FOR usage, typing IN 1..1 INBOUND fn sysml_Relations
       FILTER typing.relationship_type == "typedBy"
       FOR logical, performed IN 1..1 INBOUND usage sysml_Relations
         FILTER performed.relationship_type == "performs"
         FOR technical, spec IN 1..1 INBOUND logical sysml_Relations
           FILTER spec.relationship_type == "specializes"
           RETURN DISTINCT {function: fn.name, logical: logical.name,
                            technical: technical.name,
                            at: CONCAT(technical.source_file, ":", technical.source_line)}

rows (1, first 1)
   {"function": "GuideAscentTrajectory", "logical": "LaunchSystem", "technical": "SaturnV", "at": "apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml:159"}

A

In [14]:
az.ask("Which variants does the forest fire observation drone select?").show(row_limit=8)

Loaded AQL reference documentation from C:\Users\Christian\Desktop\arango\natural-language-service\txt2aql\prompts\aql_ref.md (104940 chars)


Successfully injected AQL reference into generation prompt.


Successfully injected AQL reference into fix prompt.


Q  Which variants does the forest fire observation drone select?

AQL
   WITH sysml_Entities, sysml_Relations
   FOR e IN sysml_Entities
     FILTER e.name == "forestFireObservationDrone"
     FOR variant, edge IN 1..2 OUTBOUND e sysml_Relations
       FILTER edge.relationship_type == "valueRef" AND variant.is_variant == true
       RETURN {feature: DOCUMENT(edge._from).name, variant: variant.name, value: variant.attributes.value.value, at: CONCAT(variant.source_file, ":", variant.source_line)}

rows (4, first 4)
   {"feature": "battery", "variant": "longDistanceBattery", "value": null, "at": "DroneModelLogical.sysml:332"}
   {"feature": "flightControl", "variant": "droneFlightControl4Engines", "value": null, "at": "DroneModelLogical.sysml:308"}
   {"feature": "body", "variant": "droneBody6Engines", "value": null, "at": "DroneModelLogical.sysml:252"}
   {"feature": "numberOfEnginesVariation", "variant": "sixEngines", "value": 6, "at": "DroneModelLogical.sysml:99"}

A  The forest fire o

## 7. Asking in English -- GraphRAG retrieval

Embed the question, vector-search entities, chunks and
community reports, expand one hop over the typed edges, answer from what came back.
Good at the descriptive and the global.

In [15]:
nl.graphrag(db, "What does the Saturn V consist of, and what is the dry mass of each stage?").show(row_limit=5)

Q  What does the Saturn V consist of, and what is the dry mass of each stage?

rows (10, first 5)
   {"entity": "TechnicalComponentsPackage::SaturnV", "score": 0.628}
   {"entity": "TechnicalComponentsPackage::S-II", "score": 0.596}
   {"entity": "TechnicalComponentsPackage::S-IC", "score": 0.564}
   {"entity": "TechnicalComponentsPackage::SaturnVInstrumentUnit", "score": 0.561}
   {"entity": "TechnicalComponentsPackage::S-IVB", "score": 0.56}

A  The Saturn V consists of the following components:

1. **Stage 1 (S-IC)**: The first stage of the Saturn V, providing the initial thrust for liftoff. Its dry mass is 137,000 kg (apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml:217).

2. **Stage 2 (S-II)**: The second stage, providing thrust in the upper atmosphere and vacuum of space. Its dry mass is 36,200 kg (apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml:201).

3. **Stage 3 (S-IVB)**: The third stage, responsible for the final push into Earth orbit and the Trans

In [16]:
nl.graphrag(db, "What are the major functional areas of the Apollo 11 model, at a high level?",
            entities=4, communities=4).show(row_limit=4)

Q  What are the major functional areas of the Apollo 11 model, at a high level?

rows (4, first 4)
   {"entity": "MissionPackage::Apollo11Mission", "score": 0.592}
   {"entity": "Apollo11Model", "score": 0.591}
   {"entity": "TechnicalComponentsPackage::ApolloCommandModule", "score": 0.58}
   {"entity": "ProgramPackage::ApolloProgram", "score": 0.577}

A  The major functional areas of the Apollo 11 model, at a high level, include:

1. **Mission Execution**: This involves the execution of the Apollo 11 mission, including the lunar landing and return (apollo-11-sysml-v2/Purpose/MissionPackage.sysml:21).

2. **Technical Components**: This area covers the detailed definitions of spacecraft parts, including the Command Module, Service Module, and Lunar Module, as well as the roles of astronauts (apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml).

3. **Functional Operations**: This focuses on the actions and requirements necessary for mission success, with a detailed breakdown o

In [17]:
# Not in the model. A confident number here would be the failure that matters --
# the reader answering from what it knows about Apollo instead of from the graph.
nl.graphrag(db, "How much did the F-1 engine cost to manufacture?").show(row_limit=3)

Q  How much did the F-1 engine cost to manufacture?

rows (10, first 3)
   {"entity": "TechnicalComponentsPackage::F-1", "score": 0.527}
   {"entity": "TechnicalComponentsPackage::S-IC::engine1", "score": 0.36}
   {"entity": "TechnicalComponentsPackage::S-IC::engine5", "score": 0.33}

A  The model does not say how much the F-1 engine cost to manufacture.



## 8. The defects in the drone models

**The variant selection is internally inconsistent.** `ForestFireObservationDrone`
binds `sixEngines` and `droneBody6Engines` and then `droneFlightControl4Engines`.

### How we parse to account for this.

A value is one of three things: a literal (`= 750 [kg]`), an expression (`= dryMass + propellantMass`),
or a reference to another element (`= A::B`). Only the third names something that exists
elsewhere in the model, so only the third gets an edge alongside the stored value. The
test is on the shape of the right-hand side; the parser never asks what the target turns
out to be, and the edge is called `valueRef` rather than `selects` because naming it
after variants would misdescribe most of what it finds.

In [18]:
print("every `valueRef` edge in the corpus, grouped by what it points at:\n")
total = 0
for r in db.aql.execute(f"""
    FOR r IN {config.RELATIONS} FILTER r.relationship_type == 'valueRef'
      LET a = DOCUMENT(r._from), b = DOCUMENT(r._to)
      COLLECT model = a.model, target = b.entity_type, is_variant = b.is_variant == true
      WITH COUNT INTO n SORT n DESC
      RETURN {{model, target, is_variant, n}}"""):
    total += r["n"]
    tag = "variant selection" if r["is_variant"] else "enumeration / plain reference"
    print(f"  {r['n']:>3}  {r['model']:<14} -> {r['target']:<18} {tag}")
n_variant = next(db.aql.execute(
    f"RETURN LENGTH(FOR r IN {config.RELATIONS} FILTER r.relationship_type == 'valueRef' "
    "FILTER DOCUMENT(r._to).is_variant == true RETURN 1)"))
print(f"\n  {total} in total, of which {n_variant} are variant selections.")
print(f"  A rule written for the drone case would have found those {n_variant} and missed "
      f"the other {total - n_variant}.")

every `valueRef` edge in the corpus, grouped by what it points at:

   24  apollo-11      -> EnumerationUsage   enumeration / plain reference
    4  drone-logical  -> EnumerationUsage   enumeration / plain reference
    3  drone-logical  -> PartUsage          variant selection
    1  drone-logical  -> AttributeUsage     variant selection

  32 in total, of which 4 are variant selections.
  A rule written for the drone case would have found those 4 and missed the other 28.


The majority of these edges are in the Apollo model, which has no variation points at
all. They are stakeholder attributes bound to enumeration literals. 

*"which stakeholders have direct influence?"* is now a traversal rather than a string comparison.

In [19]:
print("Apollo stakeholder attributes, found by the same rule:\n")
for r in db.aql.execute(f"""
    FOR r IN {config.RELATIONS} FILTER r.relationship_type == 'valueRef'
      LET a = DOCUMENT(r._from), b = DOCUMENT(r._to)
      FILTER a.model == 'apollo-11'
      LET holder = FIRST(FOR p, ed IN 1..1 INBOUND a {config.RELATIONS}
                         FILTER ed.relationship_type == 'owns' RETURN p)
      SORT holder.name, b.name LIMIT 10
      RETURN {{holder: holder.name, literal: b.entity_name,
               at: CONCAT(a.source_file, ':', a.source_line)}}"""):
    print(f"  {r['holder']:<18} -> {r['literal']:<52} {r['at'].split('/')[-1]}")

print("\nand the traversal that becomes possible because they are edges:")
print("  stakeholders with `direct` influence --")
for r in db.aql.execute(f"""
    FOR r IN {config.RELATIONS} FILTER r.relationship_type == 'valueRef'
      LET a = DOCUMENT(r._from), b = DOCUMENT(r._to)
      FILTER b.name == 'direct'
      FOR p, ed IN 1..1 INBOUND a {config.RELATIONS}
        FILTER ed.relationship_type == 'owns'
        RETURN DISTINCT {{name: p.name, at: CONCAT(p.source_file, ':', p.source_line)}}"""):
    print(f"    {r['name']:<18} {r['at'].split('/')[-1]}")

Apollo stakeholder attributes, found by the same rule:

  AmericanPublic     -> CoSMAPackage::StakeholderInfluenceKind::indirect     StakeholderPackage.sysml:165
  AmericanPublic     -> CoSMAPackage::StakeholderInfluenceLevel::medium      StakeholderPackage.sysml:164
  Apollo11Crew       -> CoSMAPackage::StakeholderInfluenceKind::direct       StakeholderPackage.sysml:53
  Apollo11Crew       -> CoSMAPackage::StakeholderInfluenceLevel::high        StakeholderPackage.sysml:52
  AstronautFamilies  -> CoSMAPackage::StakeholderInfluenceKind::indirect     StakeholderPackage.sysml:204
  AstronautFamilies  -> CoSMAPackage::StakeholderInfluenceLevel::low         StakeholderPackage.sysml:203
  FutureGenerations  -> CoSMAPackage::StakeholderInfluenceKind::indirect     StakeholderPackage.sysml:215
  FutureGenerations  -> CoSMAPackage::StakeholderInfluenceLevel::low         StakeholderPackage.sysml:214
  Geologists         -> CoSMAPackage::StakeholderInfluenceKind::indirect     StakeholderPackage.sy

### The drone models:

In [20]:
print("the selections, straight off the graph:")
for r in db.aql.execute(f"""
    FOR e IN {config.ENTITIES} FILTER e.name == 'forestFireObservationDrone'
      FOR v, edge IN 1..2 OUTBOUND e {config.RELATIONS}
        FILTER edge.relationship_type == 'valueRef' AND v.is_variant == true
        RETURN {{feature: DOCUMENT(edge._from).name, variant: v.name,
                 value: v.attributes.value.value,
                 at: CONCAT(v.source_file, ':', v.source_line)}}"""):
    print(f"  {r['feature']:<26} -> {r['variant']:<28} {r['at']}")

print("\nwhat the `is_variant` filter is for -- the same edge type, corpus-wide:")
for label, extra in [("all valueRef edges", ""),
                     ("... that are variant selections", "FILTER DOCUMENT(r._to).is_variant == true"),
                     ("... that are enumeration bindings",
                      "FILTER DOCUMENT(r._to).entity_type == 'EnumerationUsage'")]:
    n = next(db.aql.execute(f"RETURN LENGTH(FOR r IN {config.RELATIONS} "
                            f"FILTER r.relationship_type == 'valueRef' {extra} RETURN 1)"))
    print(f"  {n:>3}  {label}")
print("\n  Inside this one drone all four happen to be variants, so the filter changes")
print("  nothing here. Corpus-wide it is the difference between 4 rows and 32.")

the selections, straight off the graph:
  battery                    -> longDistanceBattery          DroneModelLogical.sysml:332
  flightControl              -> droneFlightControl4Engines   DroneModelLogical.sysml:308
  body                       -> droneBody6Engines            DroneModelLogical.sysml:252
  numberOfEnginesVariation   -> sixEngines                   DroneModelLogical.sysml:99

what the `is_variant` filter is for -- the same edge type, corpus-wide:


   32  all valueRef edges
    4  ... that are variant selections
   28  ... that are enumeration bindings

  Inside this one drone all four happen to be variants, so the filter changes
  nothing here. Corpus-wide it is the difference between 4 rows and 32.


In [21]:
nl.graphrag(db, "The forest fire observation drone selects six engines. Is its configuration "
                "internally consistent? Check the body and the flight control it also selects."
            ).show(row_limit=4)

Q  The forest fire observation drone selects six engines. Is its configuration internally consistent? Check the body and the flight control it also selects.

rows (10, first 4)
   {"entity": "DroneModelLogical::ForestFireObservationDrone::forestFireObservationDrone", "score": 0.668}
   {"entity": "DroneModelLogical::Drone_SharedAssetsSuperset::DroneFlightControl::FlightControlVariation::droneFlightControl6Engines", "score": 0.6}
   {"entity": "DroneModelLogical::ForestFireObservationDrone", "score": 0.577}
   {"entity": "DroneModelLogical::Drone_SharedAssetsSuperset::DroneBody::DroneBody_Parts::DroneBodyVariation::droneBody6Engines", "score": 0.569}

A  The configuration of the forest fire observation drone is not internally consistent. The drone selects six engines, as indicated by the variant selection `numberOfEnginesVariation = numberOfEnginesVariation::sixEngines` (DroneModelLogical.sysml:420). It also selects a body configuration that supports six engines, `body = DroneBodyVariat

In [22]:
nl.graphrag(db, "What design margin do the drone requirements leave on total mass and on "
                "battery capacity, and are the units stated?").show(row_limit=4)

Q  What design margin do the drone requirements leave on total mass and on battery capacity, and are the units stated?

rows (10, first 4)
   {"entity": "Drone_SystemArchitecture::drone", "score": 0.613}
   {"entity": "Drone_SystemRequirements", "score": 0.61}
   {"entity": "Drone_SystemRequirements::totalMass", "score": 0.608}
   {"entity": "Drone_SystemArchitecture::drone::battery", "score": 0.605}

A  The drone requirements leave no design margin on total mass, as the constraint is that the drone's total mass must be less than or equal to 750, and the drone's total mass is exactly 750 (Drone_BaseArchitecture.sysml:30, Drone_BaseArchitecture.sysml:17).

For battery capacity, the requirement is that the battery capacity must be at least 6000, and the drone's battery capacity is exactly 6000, leaving no design margin (Drone_BaseArchitecture.sysml:39, Drone_BaseArchitecture.sysml:19).

The units for total mass and battery capacity are not explicitly stated in the provided context.

